# The Last Tide: tạo art bằng SDXL

Hướng dẫn đầy đủ nằm trong `docs/MANUAL-TASKS.md`. Tóm tắt:

1. **Settings** → Accelerator: **GPU T4 x2** (hoặc P100), **Internet: On**.
2. **Add Input** → chọn dataset `last-tide-art`. Dataset này gồm `style.json`, `prompts.json` và thư mục `refs/` nếu có.
3. Sửa ô **Cấu hình** bên dưới, rồi bấm **Run All**.
4. Chạy xong, tải file `last_tide_art.zip` ở tab **Output**.

Chạy lại notebook thì **bỏ qua các ảnh đã có**. Nếu đang chạy mà bị ngắt, chỉ cần chạy lại.

In [ ]:
# ===== Cấu hình: chỉ cần sửa ô này =====
MODE = "explore"   # "explore": thử 3 style A/B/C | "assets": tạo toàn bộ asset theo style đã chọn
ONLY_IDS = []      # để trống = tạo tất cả; ví dụ ["shark", "raft_rook"] để chỉ tạo lại vài món
ONLY_CATEGORIES = []  # ví dụ ["enemies", "bosses"]
COUNT_OVERRIDE = None # ví dụ 2 để tạo ít biến thể hơn cho nhanh
IP_SCALE = 0.5     # độ bám theo ảnh mẫu trong refs/ (0.3–0.7). Cao quá thì ảnh mới bị giống y ảnh mẫu
STEPS = 30
GUIDANCE = 6.0

In [ ]:
!pip install -q "diffusers==0.31.0"

In [ ]:
import os, glob, json, zlib, shutil
from PIL import Image

found = glob.glob("/kaggle/input/**/style.json", recursive=True)
assert found, "Không thấy style.json: kiểm tra đã Add Input dataset last-tide-art chưa"
IN_DIR = os.path.dirname(found[0])
OUT_DIR = "/kaggle/working/out"
os.makedirs(OUT_DIR, exist_ok=True)

style = json.load(open(os.path.join(IN_DIR, "style.json")))
prompts = json.load(open(os.path.join(IN_DIR, "prompts.json")))

def load_refs(view):
    # Ưu tiên refs/<view>/ (ví dụ refs/portrait/), không có thì lấy ảnh trong refs/
    for d in (os.path.join(IN_DIR, "refs", view), os.path.join(IN_DIR, "refs")):
        files = sorted(f for f in glob.glob(os.path.join(d, "*")) if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp")))
        if files:
            return [Image.open(f).convert("RGB").resize((512, 512)) for f in files[:5]]
    return []

REFS = {v: load_refs(v) for v in style["views"]}
USE_IP = MODE == "assets" and any(REFS.values())
print("Input:", IN_DIR)
print("Refs:", {k: len(v) for k, v in REFS.items()}, "| IP-Adapter:", USE_IP)

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL

vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae=vae, torch_dtype=torch.float16, variant="fp16", use_safetensors=True,
).to("cuda")
pipe.enable_vae_slicing()
pipe.set_progress_bar_config(disable=True)

if USE_IP:
    pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin")

In [ ]:
def build_jobs():
    jobs = []
    if MODE == "explore":
        for cand, style_txt in style["candidates"].items():
            for s in style["explore_subjects"]:
                for v in range(COUNT_OVERRIDE or 4):
                    jobs.append(dict(id=s["id"], category=f"explore/{cand}", view=s["view"],
                                     subject=s["subject"], style=style_txt, v=v, w=1024, h=1024))
    else:
        style_txt = style["candidates"][style["chosen"]]
        for it in prompts["items"]:
            if ONLY_IDS and it["id"] not in ONLY_IDS: continue
            if ONLY_CATEGORIES and it["category"] not in ONLY_CATEGORIES: continue
            for v in range(COUNT_OVERRIDE or it.get("count", 4)):
                jobs.append(dict(id=it["id"], category=it["category"], view=it["view"], subject=it["subject"],
                                 style=style_txt, v=v, w=it.get("width", 1024), h=it.get("height", 1024)))
    return jobs

jobs = build_jobs()
todo = [j for j in jobs if not os.path.exists(f"{OUT_DIR}/{j['category']}/{j['id']}_{j['v']}.png")]
print(f"{len(jobs)} ảnh, còn {len(todo)} ảnh cần tạo (~{len(todo) * 25 // 60} phút trên T4)")

In [ ]:
import time
blank = Image.new("RGB", (512, 512), "white")
t0 = time.time()
for i, j in enumerate(todo, 1):
    path = f"{OUT_DIR}/{j['category']}/{j['id']}_{j['v']}.png"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    prompt = f"{j['style']}, {style['views'][j['view']]}, {j['subject']}"
    seed = (zlib.crc32(j["id"].encode()) + j["v"] * 7919) % (2**31)
    kwargs = dict(prompt=prompt, negative_prompt=style["negative"], width=j["w"], height=j["h"],
                  num_inference_steps=STEPS, guidance_scale=GUIDANCE,
                  generator=torch.Generator("cuda").manual_seed(seed))
    if USE_IP:
        refs = REFS.get(j["view"]) or []
        pipe.set_ip_adapter_scale(IP_SCALE if refs else 0.0)
        kwargs["ip_adapter_image"] = [refs or [blank]]
    pipe(**kwargs).images[0].save(path)
    print(f"[{i}/{len(todo)}] {path}  ({time.time() - t0:.0f}s)")
print("Xong")

In [ ]:
# Ghép ảnh mỗi nhóm thành một tấm contact sheet để duyệt cho nhanh, rồi nén thành zip
from PIL import ImageDraw
def contact_sheet(folder):
    files = sorted(glob.glob(os.path.join(folder, "*.png")))
    if not files: return
    cell, cols = 256, 4
    rows = (len(files) + cols - 1) // cols
    sheet = Image.new("RGB", (cols * cell, rows * (cell + 20)), "white")
    d = ImageDraw.Draw(sheet)
    for k, f in enumerate(files):
        im = Image.open(f).convert("RGB"); im.thumbnail((cell, cell))
        x, y = (k % cols) * cell, (k // cols) * (cell + 20)
        sheet.paste(im, (x, y)); d.text((x + 4, y + cell + 4), os.path.basename(f), fill="black")
    sheet.save(os.path.join(OUT_DIR, "_sheets", folder.replace(OUT_DIR + "/", "").replace("/", "__") + ".jpg"), quality=85)

os.makedirs(os.path.join(OUT_DIR, "_sheets"), exist_ok=True)
for folder in sorted({os.path.dirname(p) for p in glob.glob(f"{OUT_DIR}/**/*.png", recursive=True)}):
    contact_sheet(folder)

shutil.make_archive("/kaggle/working/last_tide_art", "zip", OUT_DIR)
print("Tải file /kaggle/working/last_tide_art.zip ở tab Output")